In [1]:
import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALSModel
from pyspark.sql.functions import explode, col

os.makedirs("/workspace/outputs/recommendations", exist_ok=True)

spark = (
    SparkSession.builder
    .appName("9_Recommendation_Result_Demo")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020")
    .config("spark.driver.host", "notebook")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .config("spark.driver.memory", "1g")
    .config("spark.executor.cores", "1")
    .config("spark.cores.max", "1")
    .config("spark.executor.instances", "1")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)

HDFS_RAW = "hdfs://namenode:8020/netflix-recsys/raw/ml-25m"
HDFS_MODELS = "hdfs://namenode:8020/netflix-recsys/models"
HDFS_OUTPUTS = "hdfs://namenode:8020/netflix-recsys/outputs"

movies = spark.read.parquet(f"{HDFS_RAW}/movies")
als_model = ALSModel.load(f"{HDFS_MODELS}/als_model")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/05 13:58:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

In [2]:
def recommend_for_user(user_id: int, top_n: int = 10):
    user_df = spark.createDataFrame([(user_id,)], ["userId"])
    recs = als_model.recommendForUserSubset(user_df, top_n)

    result = (
        recs
        .select("userId", explode("recommendations").alias("rec"))
        .select(
            "userId",
            col("rec.movieId").alias("movieId"),
            col("rec.rating").alias("predicted_rating")
        )
        .join(movies, "movieId", "left")
        .orderBy(col("predicted_rating").desc())
    )

    pdf = result.toPandas()
    out_path = f"/workspace/outputs/recommendations/user_{user_id}_top_{top_n}.csv"
    pdf.to_csv(out_path, index=False)
    return pdf

recommend_for_user(1, 10)

26/05/05 13:59:10 WARN TaskSetManager: Lost task 0.0 in stage 5.0 (TID 6) (172.18.0.2 executor 0): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker.py", line 1100, in main
    raise PySparkRuntimeError(
pyspark.errors.exceptions.base.PySparkRuntimeError: [PYTHON_VERSION_MISMATCH] Python in worker has different version (3, 10) than that in driver 3.11, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set.

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:784)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator

Py4JJavaError: An error occurred while calling o109.collectToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 5.0 failed 4 times, most recent failure: Lost task 0.3 in stage 5.0 (TID 12) (172.18.0.2 executor 0): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker.py", line 1100, in main
    raise PySparkRuntimeError(
pyspark.errors.exceptions.base.PySparkRuntimeError: [PYTHON_VERSION_MISMATCH] Python in worker has different version (3, 10) than that in driver 3.11, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set.

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:784)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(Pytho

In [ ]:
ratings = spark.read.parquet(f"{HDFS_RAW}/ratings")

def show_user_liked_movies(user_id: int, limit: int = 10):
    liked = (
        ratings
        .filter((col("userId") == user_id) & (col("rating") >= 4.0))
        .join(movies, "movieId", "left")
        .orderBy(col("rating").desc())
        .limit(limit)
    )
    return liked.toPandas()

show_user_liked_movies(1, 10)